# **Redefinitions of Constants**

This notebook provides a comprehensive framework for testing the ability of Large Language Models (LLMs) to answer questions about redefining constants across three difficulty levels. The generated responses are saved in .csv files for further analysis.

In [ ]:
!pip install --upgrade boto3 
!pip install --upgrade botocore
!pip install langchain

We begin by accessing a selected LLM model via AWS Bedrock. To establish access, you must provide your AWS access key credentials.

In [ ]:
# Import required libraries
import boto3
import json
from botocore.exceptions import ClientError
from langchain.prompts import PromptTemplate

# AWS Bedrock setup
client = boto3.client(
    service_name="bedrock-runtime",
    aws_access_key_id='your_aws_access_key_id',
    aws_secret_access_key='your_aws_secret_access_key',
    region_name="us-west-2"
)

# Model ID for Bedrock
model_id = "amazon.titan-text-lite-v1"

SYSTEM_PROMPT = "You are a helpful assistant."

In [ ]:
import json
import time
import random
from botocore.exceptions import ClientError  


# Invokes the model on AWS Bedrock and handles errors with retries.
# Depending on the selected model, slight adjustments may be needed.

def get_model_response(input_text, retries=5, backoff_factor=2, max_delay=60):
    native_request = {
        "inputText": input_text,
        "textGenerationConfig": {
            "maxTokenCount": 512,
            "temperature": 0.5,
            "topP": 0.9
        }
    }

    request = json.dumps(native_request)

    for attempt in range(1, retries + 1):
        try:
            response = client.invoke_model(modelId=model_id, body=request)
            model_response = json.loads(response["body"].read())

            # Extract the correct response
            output_text = model_response["results"][0]["outputText"]
            return output_text

        except (KeyError, IndexError):
            print(f"KeyError on attempt {attempt}: {model_response}")
        except ClientError as e:
            print(f"ClientError on attempt {attempt}: {e}")
        except Exception as e:
            print(f"Error on attempt {attempt}: {e}")
        
        if attempt == retries:
            print("Max retries reached. Returning None.")
            return None

        delay = min(backoff_factor * (2 ** (attempt - 1)), max_delay) + random.uniform(0, 1)
        print(f"Retrying in {delay:.2f} seconds...")
        time.sleep(delay)

In [ ]:
def get_model_answer(prompt_template, question, retries=5):
    input_text = prompt_template.format(question=question)
    return get_model_response(input_text, retries=retries)

In [ ]:
def fix_answers(llm_answers):
    llm_answers_fixed = []
    for answer in llm_answers:
        if answer is not None and 'final answer is: ' in answer:
            llm_answers_fixed.append(answer.split('final answer is: ', 1)[1])
        else:
            llm_answers_fixed.append(answer)  
    return llm_answers_fixed

# **Datasets**

The datasets used in these experiments contain information about the true values of constants, their redefined values, and the corresponding questions along with their answers related to these values. There are six datasets for the redefinition of constants: three in free-form (FF) format and three in multiple-choice (MC) format, each corresponding to a different difficulty level.

In [1]:
import os
import pandas as pd

is_kaggle = os.path.exists('/kaggle/input')

# If you're running this notebook in the Kaggle environment you need to manually upload the files contained in the 'input' folder
#   Upload these files to the Kaggle "Input" section before running the code.
#   Make sure the folder structure matches the expected directories:
#   - 'constants-level1' folder containing 'constants - level1.csv' and 'constants - level1_mc.csv'
#   - 'constants-level2' folder containing 'constants - level2.csv' and 'constants - level2_mc.csv'
#   - 'constants-level3' folder containing 'constants - level3.csv' and 'constants - level3_mc.csv'

if is_kaggle:
    input_dir = '/kaggle/input'
else:
    input_dir = os.path.join(os.getcwd(), 'input')

level1_dir = os.path.join(input_dir, "constants-level1")
level2_dir = os.path.join(input_dir, "constants-level2")
level3_dir = os.path.join(input_dir, "constants-level3")

file_level1 = ["constants - level1.csv", "constants - level1_mc.csv"]
file_level2 = ["constants - level2.csv", "constants - level2_mc.csv"]
file_level3 = ["constants - level3.csv", "constants - level3_mc.csv"]


df1 = pd.read_csv(os.path.join(level1_dir, file_level1[0]))  
df1_mc = pd.read_csv(os.path.join(level1_dir, file_level1[1]))  

df2 = pd.read_csv(os.path.join(level2_dir, file_level2[0]))  
df2_mc = pd.read_csv(os.path.join(level2_dir, file_level2[1])) 

df3 = pd.read_csv(os.path.join(level3_dir, file_level3[0]))  
df3_mc = pd.read_csv(os.path.join(level3_dir, file_level3[1])) 

In [8]:
data_level1 = [
    {
        "X": row["X"],
        "real_value": row["real_value"],
        "question": row["question"],
        "answer_real": row["answer_real"],
        "Y1": row["Y1"],
        "answer1": row["answer1"],
        "Y2": row["Y2"],
        "answer2": row["answer2"],
        "Y3": row["Y3"],
        "answer3": row["answer3"],
        "C1": row["C1"],
        "answerC1": row["answerC1"],
        "C2": row["C2"],
        "answerC2": row["answerC2"]
    }
    for _, row in df1.iterrows()
]
data_level1[:2]

[{'X': 'pi',
  'real_value': '3.14159',
  'question': 'What is the first digit of pi?',
  'answer_real': '3',
  'Y1': '4.5',
  'answer1': '4',
  'Y2': '500',
  'answer2': '5',
  'Y3': '-10',
  'answer3': '1',
  'C1': 'phi',
  'answerC1': '1',
  'C2': "the Planck's constant",
  'answerC2': '6'},
 {'X': 'e',
  'real_value': '2.71828',
  'question': 'What is the first digit of e?',
  'answer_real': '2',
  'Y1': '9',
  'answer1': '9',
  'Y2': '1300',
  'answer2': '1',
  'Y3': '1.5×10^(−12)',
  'answer3': '1',
  'C1': 'pi',
  'answerC1': '3',
  'C2': 'the Boltzmann constant',
  'answerC2': '1'}]

In [9]:
data_level2 = [
    {
        "X": row["X"],
        "real_value": row["real_value"],
        "question": row["question"],
        "answer_real": row["answer_real"],
        "Y1": row["Y1"],
        "answer1": row["answer1"],
        "Y2": row["Y2"],
        "answer2": row["answer2"],
        "Y3": row["Y3"],
        "answer3": row["answer3"],
        "C1": row["C1"],
        "answerC1": row["answerC1"],
        "C2": row["C2"],
        "answerC2": row["answerC2"]
    }
    for _, row in df2.iterrows()
]
data_level2[:2]

[{'X': 'pi',
  'real_value': '3.14159',
  'question': 'What is pi multiplied by 3?',
  'answer_real': '9.4248',
  'Y1': '4.5',
  'answer1': '13.5',
  'Y2': '500',
  'answer2': '1500',
  'Y3': '-10',
  'answer3': '-30',
  'C1': 'phi',
  'answerC1': '4.854',
  'C2': "the Planck's constant",
  'answerC2': '1.9878×10^(−33)'},
 {'X': 'e',
  'real_value': '2.71828',
  'question': 'What is e^2?',
  'answer_real': '7.389',
  'Y1': '9',
  'answer1': '81',
  'Y2': '1300',
  'answer2': '1,690,000',
  'Y3': '1.5×10^(−12)',
  'answer3': '2.25×10^(−24)',
  'C1': 'pi',
  'answerC1': '9.8696',
  'C2': 'the Boltzmann constant',
  'answerC2': '1.904×10^(−46)'}]

In [10]:
data_level3 = [
    {
        "X": row["X"],
        "real_value": row["real_value"],
        "question": row["question"],
        "answer_real": row["answer_real"],
        "Y1": row["Y1"],
        "answer1": row["answer1"],
        "Y2": row["Y2"],
        "answer2": row["answer2"],
        "Y3": row["Y3"],
        "answer3": row["answer3"],
        "C1": row["C1"],
        "answerC1": row["answerC1"],
        "C2": row["C2"],
        "answerC2": row["answerC2"]
    }
    for _, row in df3.iterrows()
]
data_level3[:2]

[{'X': 'pi',
  'real_value': '3.14159',
  'question': "What is the Earth's surface area?",
  'answer_real': '510,064,472 square kilometers',
  'Y1': '4.5',
  'answer1': '730,397,538 square kilometers',
  'Y2': '500',
  'answer2': '81,155,282,000 square kilometers',
  'Y3': '-10',
  'answer3': '−1,623,105,640 square kilometers',
  'C1': 'phi',
  'answerC1': '262,642,778 square kilometers',
  'C2': "the Planck's constant",
  'answerC2': '1.08×10^(−25) square kilometers'},
 {'X': 'e',
  'real_value': '2.71828',
  'question': 'If a population grows continuously at a rate of 5% per year, by what factor will it increase in 10 years?',
  'answer_real': '1.649',
  'Y1': '9',
  'answer1': '3',
  'Y2': '1300',
  'answer2': '36.06',
  'Y3': '1.5×10^(−12)',
  'answer3': '1.22×10^(−6)',
  'C1': 'pi',
  'answerC1': '1.77245',
  'C2': 'the Boltzmann constant',
  'answerC2': '3.714×10^(−12) '}]

In [11]:
data_level1_mc = [
    {
        "X": row["X"],
        "real_value": row["real_value"],
        "question": row["question"],
        "answer_real": row["answer_real"],
        "Y1": row["Y1"],
        "A1": row["A1"],
        "B1": row["B1"],
        "C1": row["C1"],
        "D1": row["D1"],
        "answer1": row["answer1"],
        "answer1_real": row["answer1_real"],
        "Y2": row["Y2"],
        "A2": row["A2"],
        "B2": row["B2"],
        "C2": row["C2"],
        "D2": row["D2"],
        "answer2": row["answer2"],
        "answer2_real": row["answer2_real"],
        "Y3": row["Y3"],
        "A3": row["A3"],
        "B3": row["B3"],
        "C3": row["C3"],
        "D3": row["D3"],
        "answer3": row["answer3"],
        "answer3_real": row["answer3_real"],
        "YC1": row["YC1"],
        "AC1": row["AC1"],
        "BC1": row["BC1"],
        "CC1": row["CC1"],
        "DC1": row["DC1"],
        "answerC1": row["answerC1"],
        "answerC1_real": row["answerC1_real"],
        "YC2": row["YC2"],
        "AC2": row["AC2"],
        "BC2": row["BC2"],
        "CC2": row["CC2"],
        "DC2": row["DC2"],
        "answerC2": row["answerC2"],
        "answerC2_real": row["answerC2_real"]
    }
    for _, row in df1_mc.iterrows()
]
data_level1_mc[:2]

[{'X': 'pi',
  'real_value': '3.14159',
  'question': 'What is the first digit of pi?',
  'answer_real': '3',
  'Y1': '4.5',
  'A1': '2',
  'B1': '3',
  'C1': '4',
  'D1': '5',
  'answer1': 'C',
  'answer1_real': 'B',
  'Y2': '500',
  'A2': '2',
  'B2': '3',
  'C2': '4',
  'D2': '5',
  'answer2': 'D',
  'answer2_real': 'B',
  'Y3': '-10',
  'A3': '1',
  'B3': '2',
  'C3': '3',
  'D3': '4',
  'answer3': 'A',
  'answer3_real': 'C',
  'YC1': 'phi',
  'AC1': '1',
  'BC1': '2',
  'CC1': '3',
  'DC1': '4',
  'answerC1': 'A',
  'answerC1_real': 'C',
  'YC2': "the Planck's constant",
  'AC2': '3',
  'BC2': '4',
  'CC2': '5',
  'DC2': '6',
  'answerC2': 'D',
  'answerC2_real': 'A'},
 {'X': 'e',
  'real_value': '2.71828',
  'question': 'What is the first digit of e?',
  'answer_real': '2',
  'Y1': '9',
  'A1': '2',
  'B1': '4',
  'C1': '7',
  'D1': '9',
  'answer1': 'D',
  'answer1_real': 'A',
  'Y2': '1300',
  'A2': '1',
  'B2': '2',
  'C2': '3',
  'D2': '4',
  'answer2': 'A',
  'answer2_real':

In [12]:
data_level2_mc = [
    {
        "X": row["X"],
        "real_value": row["real_value"],
        "question": row["question"],
        "answer_real": row["answer_real"],
        "Y1": row["Y1"],
        "A1": row["A1"],
        "B1": row["B1"],
        "C1": row["C1"],
        "D1": row["D1"],
        "answer1": row["answer1"],
        "answer1_real": row["answer1_real"],
        "Y2": row["Y2"],
        "A2": row["A2"],
        "B2": row["B2"],
        "C2": row["C2"],
        "D2": row["D2"],
        "answer2": row["answer2"],
        "answer2_real": row["answer2_real"],
        "Y3": row["Y3"],
        "A3": row["A3"],
        "B3": row["B3"],
        "C3": row["C3"],
        "D3": row["D3"],
        "answer3": row["answer3"],
        "answer3_real": row["answer3_real"],
        "YC1": row["YC1"],
        "AC1": row["AC1"],
        "BC1": row["BC1"],
        "CC1": row["CC1"],
        "DC1": row["DC1"],
        "answerC1": row["answerC1"],
        "answerC1_real": row["answerC1_real"],
        "YC2": row["YC2"],
        "AC2": row["AC2"],
        "BC2": row["BC2"],
        "CC2": row["CC2"],
        "DC2": row["DC2"],
        "answerC2": row["answerC2"],
        "answerC2_real": row["answerC2_real"]
    }
    for _, row in df2_mc.iterrows()
]
data_level2_mc[:2]

[{'X': 'pi',
  'real_value': '3.14159',
  'question': 'What is pi multiplied by 3?',
  'answer_real': '9.4248',
  'Y1': '4.5',
  'A1': '9',
  'B1': '9.4248',
  'C1': '13.5',
  'D1': '15',
  'answer1': 'C',
  'answer1_real': 'B',
  'Y2': '500',
  'A2': '9',
  'B2': '9.4248',
  'C2': '500',
  'D2': '1500',
  'answer2': 'D',
  'answer2_real': 'B',
  'Y3': '-10',
  'A3': '9',
  'B3': '9.4248',
  'C3': '-30',
  'D3': '30',
  'answer3': 'C',
  'answer3_real': 'B',
  'YC1': 'phi',
  'AC1': '1.61803',
  'BC1': '4.854',
  'CC1': '3.14159',
  'DC1': '9.4248',
  'answerC1': 'B',
  'answerC1_real': 'D',
  'YC2': "the Planck's constant",
  'AC2': '1.9878',
  'BC2': '1.9878×10^(−33)',
  'CC2': '3.14159',
  'DC2': '9.4248',
  'answerC2': 'B',
  'answerC2_real': 'D'},
 {'X': 'e',
  'real_value': '2.71828',
  'question': 'What is e^2?',
  'answer_real': '7.389',
  'Y1': '9',
  'A1': '7.389',
  'B1': '9',
  'C1': '18',
  'D1': '81',
  'answer1': 'D',
  'answer1_real': 'A',
  'Y2': '1300',
  'A2': '7.389

In [13]:
data_level3_mc = [
    {
        "X": row["X"],
        "real_value": row["real_value"],
        "question": row["question"],
        "answer_real": row["answer_real"],
        "Y1": row["Y1"],
        "A1": row["A1"],
        "B1": row["B1"],
        "C1": row["C1"],
        "D1": row["D1"],
        "answer1": row["answer1"],
        "answer1_real": row["answer1_real"],
        "Y2": row["Y2"],
        "A2": row["A2"],
        "B2": row["B2"],
        "C2": row["C2"],
        "D2": row["D2"],
        "answer2": row["answer2"],
        "answer2_real": row["answer2_real"],
        "Y3": row["Y3"],
        "A3": row["A3"],
        "B3": row["B3"],
        "C3": row["C3"],
        "D3": row["D3"],
        "answer3": row["answer3"],
        "answer3_real": row["answer3_real"],
        "YC1": row["YC1"],
        "AC1": row["AC1"],
        "BC1": row["BC1"],
        "CC1": row["CC1"],
        "DC1": row["DC1"],
        "answerC1": row["answerC1"],
        "answerC1_real": row["answerC1_real"],
        "YC2": row["YC2"],
        "AC2": row["AC2"],
        "BC2": row["BC2"],
        "CC2": row["CC2"],
        "DC2": row["DC2"],
        "answerC2": row["answerC2"],
        "answerC2_real": row["answerC2_real"]
    }
    for _, row in df3_mc.iterrows()
]
data_level3_mc[:2]

[{'X': 'pi',
  'real_value': '3.14159',
  'question': "What is the Earth's surface area?",
  'answer_real': '510,064,472 square kilometers',
  'Y1': '4.5',
  'A1': '510,064,472 square kilometers',
  'B1': '730,397,538 square kilometers',
  'C1': '730,397,538 square meters',
  'D1': '510,064,472 square meters',
  'answer1': 'B',
  'answer1_real': 'A',
  'Y2': '500',
  'A2': '510,064,472 square meters',
  'B2': '510,064,472 square kilometers',
  'C2': '81,155,282 square kilometers',
  'D2': '81,155,282,000 square kilometers',
  'answer2': 'D',
  'answer2_real': 'B',
  'Y3': '-10',
  'A3': '510,064,472 square meters',
  'B3': '510,064,472 square kilometers',
  'C3': '−1,623,105,640 square kilometers',
  'D3': '-510,064,472 square kilometers',
  'answer3': 'C',
  'answer3_real': 'B',
  'YC1': 'phi',
  'AC1': '153,968,467 square kilometers',
  'BC1': '262,642,778 square kilometers',
  'CC1': '510,064,472 square meters',
  'DC1': '510,064,472 square kilometers',
  'answerC1': 'B',
  'answerC

# **No Redefinition**

We first ask these questions without redefining the constants to assess the model's baseline knowledge. In this notebook, for the no-redefinition phase, we focus on the free-form format using zero-shot prompting. Other experiments exploring different prompting techniques for both free-form and multiple-choice formats can be found in no_redefinition.ipynb.

In [ ]:
template = """
Answer the following question:
{question}

End the response with the phrase "The final answer is: " followed only by the correct result, with no additional text or commentary.
"""

prompt_template = PromptTemplate(
    input_variables=["question"],
    template=template
)

In [ ]:
answers1 = []

for row in data_level1:
    question = row['question']
    answers1.append(get_model_answer(prompt_template, question))

In [ ]:
answers2 = []

for row in data_level2:
    question = row['question']
    answers2.append(get_model_answer(prompt_template, question))

In [ ]:
answers3 = []

for row in data_level3:
    question = row['question']
    answers3.append(get_model_answer(prompt_template, question))

In [ ]:
answers1_fixed = fix_answers(answers1)
answers2_fixed = fix_answers(answers2)
answers3_fixed = fix_answers(answers3)

# **Redefinition**

We conduct experiments for both free-form and multiple-choice formats varying by redefinition types, redefinition levels, question difficulty levels, and prompting techniques. 

## **Free Form**

### **Zero-Shot**

In [ ]:
template_redefinition = """
Redefine {X} as {Y}. {question}

End the response with the phrase "The final answer is: " followed only by the correct result, with no additional text or commentary.
"""

from langchain.prompts import PromptTemplate

prompt_template_redefinition = PromptTemplate(
    input_variables=["question", "X", "Y"],
    template=template_redefinition
)

In [ ]:
def get_model_answer_redefinition(prompt_template, question, X, Y, retries=5):
    input_text = prompt_template.format(question=question, X=X, Y=Y)
    return get_model_response(input_text, retries=retries)

In [ ]:
answers1_Y1_redefinition, answers1_Y2_redefinition, answers1_Y3_redefinition, answers1_C1_redefinition, answers1_C2_redefinition = [], [], [], [], []
answers2_Y1_redefinition, answers2_Y2_redefinition, answers2_Y3_redefinition, answers2_C1_redefinition, answers2_C2_redefinition = [], [], [], [], []
answers3_Y1_redefinition, answers3_Y2_redefinition, answers3_Y3_redefinition, answers3_C1_redefinition, answers3_C2_redefinition = [], [], [], [], []

In [ ]:
for row in data_level1:
    question = row['question']
    X = row['X']
    Y1 = row['Y1']
    Y2 = row['Y2']
    Y3 = row['Y3']
    C1 = row['C1']
    C2 = row['C2']
    
    answers1_Y1_redefinition.append(get_model_answer_redefinition(prompt_template_redefinition, question, X , Y1))
    answers1_Y2_redefinition.append(get_model_answer_redefinition(prompt_template_redefinition, question, X , Y2))
    answers1_Y3_redefinition.append(get_model_answer_redefinition(prompt_template_redefinition, question, X , Y3))
    answers1_C1_redefinition.append(get_model_answer_redefinition(prompt_template_redefinition, question, X , C1))
    answers1_C2_redefinition.append(get_model_answer_redefinition(prompt_template_redefinition, question, X , C2))

In [ ]:
for row in data_level2:
    question = row['question']
    X = row['X']
    Y1 = row['Y1']
    Y2 = row['Y2']
    Y3 = row['Y3']
    C1 = row['C1']
    C2 = row['C2']
    
    answers2_Y1_redefinition.append(get_model_answer_redefinition(prompt_template_redefinition, question, X , Y1))
    answers2_Y2_redefinition.append(get_model_answer_redefinition(prompt_template_redefinition, question, X , Y2))
    answers2_Y3_redefinition.append(get_model_answer_redefinition(prompt_template_redefinition, question, X , Y3))
    answers2_C1_redefinition.append(get_model_answer_redefinition(prompt_template_redefinition, question, X , C1))
    answers2_C2_redefinition.append(get_model_answer_redefinition(prompt_template_redefinition, question, X , C2))

In [ ]:
for row in data_level3:
    question = row['question']
    X = row['X']
    Y1 = row['Y1']
    Y2 = row['Y2']
    Y3 = row['Y3']
    C1 = row['C1']
    C2 = row['C2']
    
    answers3_Y1_redefinition.append(get_model_answer_redefinition(prompt_template_redefinition, question, X , Y1))
    answers3_Y2_redefinition.append(get_model_answer_redefinition(prompt_template_redefinition, question, X , Y2))
    answers3_Y3_redefinition.append(get_model_answer_redefinition(prompt_template_redefinition, question, X , Y3))
    answers3_C1_redefinition.append(get_model_answer_redefinition(prompt_template_redefinition, question, X , C1))
    answers3_C2_redefinition.append(get_model_answer_redefinition(prompt_template_redefinition, question, X , C2))

In [ ]:
answers1_Y1_redefinition_fixed = fix_answers(answers1_Y1_redefinition)
answers1_Y2_redefinition_fixed = fix_answers(answers1_Y2_redefinition)
answers1_Y3_redefinition_fixed = fix_answers(answers1_Y3_redefinition)
answers1_C1_redefinition_fixed = fix_answers(answers1_C1_redefinition)
answers1_C2_redefinition_fixed = fix_answers(answers1_C2_redefinition)

In [ ]:
answers2_Y1_redefinition_fixed = fix_answers(answers2_Y1_redefinition)
answers2_Y2_redefinition_fixed = fix_answers(answers2_Y2_redefinition)
answers2_Y3_redefinition_fixed = fix_answers(answers2_Y3_redefinition)
answers2_C1_redefinition_fixed = fix_answers(answers2_C1_redefinition)
answers2_C2_redefinition_fixed = fix_answers(answers2_C2_redefinition)

In [ ]:
answers3_Y1_redefinition_fixed = fix_answers(answers3_Y1_redefinition)
answers3_Y2_redefinition_fixed = fix_answers(answers3_Y2_redefinition)
answers3_Y3_redefinition_fixed = fix_answers(answers3_Y3_redefinition)
answers3_C1_redefinition_fixed = fix_answers(answers3_C1_redefinition)
answers3_C2_redefinition_fixed = fix_answers(answers3_C2_redefinition)

### **Chain-of-Thought (CoT)**

In [ ]:
template_redefinition_cot = """
Redefine {X} as {Y}. {question}

Let's think step by step. 

End the response with the phrase "The final answer is: " followed only by the correct result, with no additional text or commentary.
"""

from langchain.prompts import PromptTemplate

prompt_template_redefinition_cot = PromptTemplate(
    input_variables=["question", "X", "Y"],
    template=template_redefinition_cot
)

In [ ]:
answers1_Y1_redefinition_cot, answers1_Y2_redefinition_cot, answers1_Y3_redefinition_cot, answers1_C1_redefinition_cot, answers1_C2_redefinition_cot = [], [], [], [], []
answers2_Y1_redefinition_cot, answers2_Y2_redefinition_cot, answers2_Y3_redefinition_cot, answers2_C1_redefinition_cot, answers2_C2_redefinition_cot = [], [], [], [], []
answers3_Y1_redefinition_cot, answers3_Y2_redefinition_cot, answers3_Y3_redefinition_cot, answers3_C1_redefinition_cot, answers3_C2_redefinition_cot = [], [], [], [], []

In [ ]:
for row in data_level1:
    question = row['question']
    X = row['X']
    Y1 = row['Y1']
    Y2 = row['Y2']
    Y3 = row['Y3']
    C1 = row['C1']
    C2 = row['C2']
    
    answers1_Y1_redefinition_cot.append(get_model_answer_redefinition(prompt_template_redefinition_cot, question, X , Y1))
    answers1_Y2_redefinition_cot.append(get_model_answer_redefinition(prompt_template_redefinition_cot, question, X , Y2))
    answers1_Y3_redefinition_cot.append(get_model_answer_redefinition(prompt_template_redefinition_cot, question, X , Y3))
    answers1_C1_redefinition_cot.append(get_model_answer_redefinition(prompt_template_redefinition_cot, question, X , C1))
    answers1_C2_redefinition_cot.append(get_model_answer_redefinition(prompt_template_redefinition_cot, question, X , C2))

In [ ]:
for row in data_level2:
    question = row['question']
    X = row['X']
    Y1 = row['Y1']
    Y2 = row['Y2']
    Y3 = row['Y3']
    C1 = row['C1']
    C2 = row['C2']
    
    answers2_Y1_redefinition_cot.append(get_model_answer_redefinition(prompt_template_redefinition_cot, question, X , Y1))
    answers2_Y2_redefinition_cot.append(get_model_answer_redefinition(prompt_template_redefinition_cot, question, X , Y2))
    answers2_Y3_redefinition_cot.append(get_model_answer_redefinition(prompt_template_redefinition_cot, question, X , Y3))
    answers2_C1_redefinition_cot.append(get_model_answer_redefinition(prompt_template_redefinition_cot, question, X , C1))
    answers2_C2_redefinition_cot.append(get_model_answer_redefinition(prompt_template_redefinition_cot, question, X , C2))

In [ ]:
for row in data_level3:
    question = row['question']
    X = row['X']
    Y1 = row['Y1']
    Y2 = row['Y2']
    Y3 = row['Y3']
    C1 = row['C1']
    C2 = row['C2']
    
    answers3_Y1_redefinition_cot.append(get_model_answer_redefinition(prompt_template_redefinition_cot, question, X , Y1))
    answers3_Y2_redefinition_cot.append(get_model_answer_redefinition(prompt_template_redefinition_cot, question, X , Y2))
    answers3_Y3_redefinition_cot.append(get_model_answer_redefinition(prompt_template_redefinition_cot, question, X , Y3))
    answers3_C1_redefinition_cot.append(get_model_answer_redefinition(prompt_template_redefinition_cot, question, X , C1))
    answers3_C2_redefinition_cot.append(get_model_answer_redefinition(prompt_template_redefinition_cot, question, X , C2))

In [ ]:
answers1_Y1_redefinition_cot_fixed = fix_answers(answers1_Y1_redefinition_cot)
answers1_Y2_redefinition_cot_fixed = fix_answers(answers1_Y2_redefinition_cot)
answers1_Y3_redefinition_cot_fixed = fix_answers(answers1_Y3_redefinition_cot)
answers1_C1_redefinition_cot_fixed = fix_answers(answers1_C1_redefinition_cot)
answers1_C2_redefinition_cot_fixed = fix_answers(answers1_C2_redefinition_cot)

In [ ]:
answers2_Y1_redefinition_cot_fixed = fix_answers(answers2_Y1_redefinition_cot)
answers2_Y2_redefinition_cot_fixed = fix_answers(answers2_Y2_redefinition_cot)
answers2_Y3_redefinition_cot_fixed = fix_answers(answers2_Y3_redefinition_cot)
answers2_C1_redefinition_cot_fixed = fix_answers(answers2_C1_redefinition_cot)
answers2_C2_redefinition_cot_fixed = fix_answers(answers2_C2_redefinition_cot)

In [ ]:
answers3_Y1_redefinition_cot_fixed = fix_answers(answers3_Y1_redefinition_cot)
answers3_Y2_redefinition_cot_fixed = fix_answers(answers3_Y2_redefinition_cot)
answers3_Y3_redefinition_cot_fixed = fix_answers(answers3_Y3_redefinition_cot)
answers3_C1_redefinition_cot_fixed = fix_answers(answers3_C1_redefinition_cot)
answers3_C2_redefinition_cot_fixed = fix_answers(answers3_C2_redefinition_cot)

### **Few-Shot**

In [ ]:
template_redefinition_f = """
Redefine {X} as {Y}. {question}

Here are some examples of similar questions with their correct answers:

Question: Redefine tau as 9. What is 10*tau?
Answer: 90

Question: Redefine one as 10. What is one percent of 50?
Answer: 5

Question: Redefine vacuum magnetic permeability as 4.56×10^(−7) N/A^2. What is the first digit of vacuum magnetic permeability?
Answer: 4

End the response with the phrase "The final answer is: " followed only by the correct result, with no additional text or commentary.
"""

from langchain.prompts import PromptTemplate

prompt_template_redefinition_f = PromptTemplate(
    input_variables=["question", "X", "Y"],
    template=template_redefinition_f
)

In [ ]:
answers1_Y1_redefinition_f, answers1_Y2_redefinition_f, answers1_Y3_redefinition_f, answers1_C1_redefinition_f, answers1_C2_redefinition_f = [], [], [], [], []
answers2_Y1_redefinition_f, answers2_Y2_redefinition_f, answers2_Y3_redefinition_f, answers2_C1_redefinition_f, answers2_C2_redefinition_f = [], [], [], [], []
answers3_Y1_redefinition_f, answers3_Y2_redefinition_f, answers3_Y3_redefinition_f, answers3_C1_redefinition_f, answers3_C2_redefinition_f = [], [], [], [], []

In [ ]:
for row in data_level1:
    question = row['question']
    X = row['X']
    Y1 = row['Y1']
    Y2 = row['Y2']
    Y3 = row['Y3']
    C1 = row['C1']
    C2 = row['C2']
    
    answers1_Y1_redefinition_f.append(get_model_answer_redefinition(prompt_template_redefinition_f, question, X, Y1))
    answers1_Y2_redefinition_f.append(get_model_answer_redefinition(prompt_template_redefinition_f, question, X, Y2))
    answers1_Y3_redefinition_f.append(get_model_answer_redefinition(prompt_template_redefinition_f, question, X, Y3))
    answers1_C1_redefinition_f.append(get_model_answer_redefinition(prompt_template_redefinition_f, question, X, C1))
    answers1_C2_redefinition_f.append(get_model_answer_redefinition(prompt_template_redefinition_f, question, X, C2))

In [ ]:
for row in data_level2:
    question = row['question']
    X = row['X']
    Y1 = row['Y1']
    Y2 = row['Y2']
    Y3 = row['Y3']
    C1 = row['C1']
    C2 = row['C2']
    
    answers2_Y1_redefinition_f.append(get_model_answer_redefinition(prompt_template_redefinition_f, question, X, Y1))
    answers2_Y2_redefinition_f.append(get_model_answer_redefinition(prompt_template_redefinition_f, question, X, Y2))
    answers2_Y3_redefinition_f.append(get_model_answer_redefinition(prompt_template_redefinition_f, question, X, Y3))
    answers2_C1_redefinition_f.append(get_model_answer_redefinition(prompt_template_redefinition_f, question, X, C1))
    answers2_C2_redefinition_f.append(get_model_answer_redefinition(prompt_template_redefinition_f, question, X, C2))

In [ ]:
for row in data_level3:
    question = row['question']
    X = row['X']
    Y1 = row['Y1']
    Y2 = row['Y2']
    Y3 = row['Y3']
    C1 = row['C1']
    C2 = row['C2']
    
    answers3_Y1_redefinition_f.append(get_model_answer_redefinition(prompt_template_redefinition_f, question, X, Y1))
    answers3_Y2_redefinition_f.append(get_model_answer_redefinition(prompt_template_redefinition_f, question, X, Y2))
    answers3_Y3_redefinition_f.append(get_model_answer_redefinition(prompt_template_redefinition_f, question, X, Y3))
    answers3_C1_redefinition_f.append(get_model_answer_redefinition(prompt_template_redefinition_f, question, X, C1))
    answers3_C2_redefinition_f.append(get_model_answer_redefinition(prompt_template_redefinition_f, question, X, C2))

In [ ]:
answers1_Y1_redefinition_f_fixed = fix_answers(answers1_Y1_redefinition_f)
answers1_Y2_redefinition_f_fixed = fix_answers(answers1_Y2_redefinition_f)
answers1_Y3_redefinition_f_fixed = fix_answers(answers1_Y3_redefinition_f)
answers1_C1_redefinition_f_fixed = fix_answers(answers1_C1_redefinition_f)
answers1_C2_redefinition_f_fixed = fix_answers(answers1_C2_redefinition_f)

In [ ]:
answers2_Y1_redefinition_f_fixed = fix_answers(answers2_Y1_redefinition_f)
answers2_Y2_redefinition_f_fixed = fix_answers(answers2_Y2_redefinition_f)
answers2_Y3_redefinition_f_fixed = fix_answers(answers2_Y3_redefinition_f)
answers2_C1_redefinition_f_fixed = fix_answers(answers2_C1_redefinition_f)
answers2_C2_redefinition_f_fixed = fix_answers(answers2_C2_redefinition_f)

In [ ]:
answers3_Y1_redefinition_f_fixed = fix_answers(answers3_Y1_redefinition_f)
answers3_Y2_redefinition_f_fixed = fix_answers(answers3_Y2_redefinition_f)
answers3_Y3_redefinition_f_fixed = fix_answers(answers3_Y3_redefinition_f)
answers3_C1_redefinition_f_fixed = fix_answers(answers3_C1_redefinition_f)
answers3_C2_redefinition_f_fixed = fix_answers(answers3_C2_redefinition_f)

## **Multiple Choice**

### **Zero-Shot**

In [ ]:
template_redefinition_mc = """
Redefine {X} as {Y}. Choose A, B, C, or D to answer the question.

Question: {question}
A: {A}
B: {B}
C: {C}
D: {D}

Provide only the letter corresponding to the correct answer: "A", "B", "C", or "D".
End the response with the phrase "The final answer is: " followed by the correct letter, with no additional text or commentary.
"""

from langchain.prompts import PromptTemplate

prompt_template_redefinition_mc = PromptTemplate(
    input_variables=["question", "X", "Y", "A", "B", "C", "D"],
    template=template_redefinition_mc
)

In [ ]:
def get_model_answer_redefinition_mc(prompt_template, question, X, Y, A, B, C, D, retries=5):
    input_text = prompt_template.format(question=question, X=X, Y=Y, A=A, B=B, C=C, D=D)
    return get_model_response(input_text, retries=retries)

In [ ]:
answers1_Y1_redefinition_mc, answers1_Y2_redefinition_mc, answers1_Y3_redefinition_mc, answers1_C1_redefinition_mc, answers1_C2_redefinition_mc = [], [], [], [], []
answers2_Y1_redefinition_mc, answers2_Y2_redefinition_mc, answers2_Y3_redefinition_mc, answers2_C1_redefinition_mc, answers2_C2_redefinition_mc = [], [], [], [], []
answers3_Y1_redefinition_mc, answers3_Y2_redefinition_mc, answers3_Y3_redefinition_mc, answers3_C1_redefinition_mc, answers3_C2_redefinition_mc = [], [], [], [], []

In [ ]:
for row in data_level1_mc:
    question = row['question']
    X = row['X']
    Y1 = row['Y1']
    A1, B1, C1, D1 = row['A1'], row['B1'], row['C1'], row['D1']
    Y2 = row['Y2']
    A2, B2, C2, D2 = row['A2'], row['B2'], row['C2'], row['D2']
    Y3 = row['Y3']
    A3, B3, C3, D3 = row['A3'], row['B3'], row['C3'], row['D3']
    YC1 = row['YC1']
    AC1, BC1, CC1, DC1 = row['AC1'], row['BC1'], row['CC1'], row['DC1']
    YC2 = row['YC2']
    AC2, BC2, CC2, DC2 = row['AC2'], row['BC2'], row['CC2'], row['DC2']

    answers1_Y1_redefinition_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc, question, X, Y1, A1, B1, C1, D1))
    answers1_Y2_redefinition_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc, question, X, Y2, A2, B2, C2, D2))
    answers1_Y3_redefinition_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc, question, X, Y3, A3, B3, C3, D3))
    answers1_C1_redefinition_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc, question, X, YC1, AC1, BC1, CC1, DC1))
    answers1_C2_redefinition_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc, question, X, YC2, AC2, BC2, CC2, DC2))

In [ ]:
for row in data_level2_mc:
    question = row['question']
    X = row['X']
    Y1 = row['Y1']
    A1, B1, C1, D1 = row['A1'], row['B1'], row['C1'], row['D1']
    Y2 = row['Y2']
    A2, B2, C2, D2 = row['A2'], row['B2'], row['C2'], row['D2']
    Y3 = row['Y3']
    A3, B3, C3, D3 = row['A3'], row['B3'], row['C3'], row['D3']
    YC1 = row['YC1']
    AC1, BC1, CC1, DC1 = row['AC1'], row['BC1'], row['CC1'], row['DC1']
    YC2 = row['YC2']
    AC2, BC2, CC2, DC2 = row['AC2'], row['BC2'], row['CC2'], row['DC2']

    answers2_Y1_redefinition_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc, question, X, Y1, A1, B1, C1, D1))
    answers2_Y2_redefinition_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc, question, X, Y2, A2, B2, C2, D2))
    answers2_Y3_redefinition_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc, question, X, Y3, A3, B3, C3, D3))
    answers2_C1_redefinition_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc, question, X, YC1, AC1, BC1, CC1, DC1))
    answers2_C2_redefinition_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc, question, X, YC2, AC2, BC2, CC2, DC2))

In [ ]:
for row in data_level3_mc:
    question = row['question']
    X = row['X']
    Y1 = row['Y1']
    A1, B1, C1, D1 = row['A1'], row['B1'], row['C1'], row['D1']
    Y2 = row['Y2']
    A2, B2, C2, D2 = row['A2'], row['B2'], row['C2'], row['D2']
    Y3 = row['Y3']
    A3, B3, C3, D3 = row['A3'], row['B3'], row['C3'], row['D3']
    YC1 = row['YC1']
    AC1, BC1, CC1, DC1 = row['AC1'], row['BC1'], row['CC1'], row['DC1']
    YC2 = row['YC2']
    AC2, BC2, CC2, DC2 = row['AC2'], row['BC2'], row['CC2'], row['DC2']

    answers3_Y1_redefinition_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc, question, X, Y1, A1, B1, C1, D1))
    answers3_Y2_redefinition_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc, question, X, Y2, A2, B2, C2, D2))
    answers3_Y3_redefinition_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc, question, X, Y3, A3, B3, C3, D3))
    answers3_C1_redefinition_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc, question, X, YC1, AC1, BC1, CC1, DC1))
    answers3_C2_redefinition_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc, question, X, YC2, AC2, BC2, CC2, DC2))

In [ ]:
answers1_Y1_redefinition_mc_fixed = fix_answers(answers1_Y1_redefinition_mc)
answers1_Y2_redefinition_mc_fixed = fix_answers(answers1_Y2_redefinition_mc)
answers1_Y3_redefinition_mc_fixed = fix_answers(answers1_Y3_redefinition_mc)
answers1_C1_redefinition_mc_fixed = fix_answers(answers1_C1_redefinition_mc)
answers1_C2_redefinition_mc_fixed = fix_answers(answers1_C2_redefinition_mc)

In [ ]:
answers2_Y1_redefinition_mc_fixed = fix_answers(answers2_Y1_redefinition_mc)
answers2_Y2_redefinition_mc_fixed = fix_answers(answers2_Y2_redefinition_mc)
answers2_Y3_redefinition_mc_fixed = fix_answers(answers2_Y3_redefinition_mc)
answers2_C1_redefinition_mc_fixed = fix_answers(answers2_C1_redefinition_mc)
answers2_C2_redefinition_mc_fixed = fix_answers(answers2_C2_redefinition_mc)

In [ ]:
answers3_Y1_redefinition_mc_fixed = fix_answers(answers3_Y1_redefinition_mc)
answers3_Y2_redefinition_mc_fixed = fix_answers(answers3_Y2_redefinition_mc)
answers3_Y3_redefinition_mc_fixed = fix_answers(answers3_Y3_redefinition_mc)
answers3_C1_redefinition_mc_fixed = fix_answers(answers3_C1_redefinition_mc)
answers3_C2_redefinition_mc_fixed = fix_answers(answers3_C2_redefinition_mc)

### **Chain-of-Thought (CoT)**

In [ ]:
template_redefinition_mc_cot = """
Redefine {X} as {Y}. Choose A, B, C, or D to answer the question.

Question: {question}
A: {A}
B: {B}
C: {C}
D: {D}

Let's think step by step. 

Provide only the letter corresponding to the correct answer: "A", "B", "C", or "D".
End the response with the phrase "The final answer is: " followed by the correct letter, with no additional text or commentary.
"""

prompt_template_redefinition_mc_cot = PromptTemplate(
    input_variables=["question", "X", "Y", "A", "B", "C", "D"],
    template=template_redefinition_mc_cot
)

In [ ]:
answers1_Y1_redefinition_cot_mc, answers1_Y2_redefinition_cot_mc, answers1_Y3_redefinition_cot_mc, answers1_C1_redefinition_cot_mc, answers1_C2_redefinition_cot_mc = [], [], [], [], []
answers2_Y1_redefinition_cot_mc, answers2_Y2_redefinition_cot_mc, answers2_Y3_redefinition_cot_mc, answers2_C1_redefinition_cot_mc, answers2_C2_redefinition_cot_mc = [], [], [], [], []
answers3_Y1_redefinition_cot_mc, answers3_Y2_redefinition_cot_mc, answers3_Y3_redefinition_cot_mc, answers3_C1_redefinition_cot_mc, answers3_C2_redefinition_cot_mc = [], [], [], [], []

In [ ]:
for row in data_level1_mc:
    question = row['question']
    X = row['X']
    Y1 = row['Y1']
    A1, B1, C1, D1 = row['A1'], row['B1'], row['C1'], row['D1']
    Y2 = row['Y2']
    A2, B2, C2, D2 = row['A2'], row['B2'], row['C2'], row['D2']
    Y3 = row['Y3']
    A3, B3, C3, D3 = row['A3'], row['B3'], row['C3'], row['D3']
    YC1 = row['YC1']
    AC1, BC1, CC1, DC1 = row['AC1'], row['BC1'], row['CC1'], row['DC1']
    YC2 = row['YC2']
    AC2, BC2, CC2, DC2 = row['AC2'], row['BC2'], row['CC2'], row['DC2']

    answers1_Y1_redefinition_cot_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_cot, question, X, Y1, A1, B1, C1, D1))
    answers1_Y2_redefinition_cot_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_cot, question, X, Y2, A2, B2, C2, D2))
    answers1_Y3_redefinition_cot_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_cot, question, X, Y3, A3, B3, C3, D3))
    answers1_C1_redefinition_cot_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_cot, question, X, YC1, AC1, BC1, CC1, DC1))
    answers1_C2_redefinition_cot_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_cot, question, X, YC2, AC2, BC2, CC2, DC2))

In [ ]:
for row in data_level2_mc:
    question = row['question']
    X = row['X']
    Y1 = row['Y1']
    A1, B1, C1, D1 = row['A1'], row['B1'], row['C1'], row['D1']
    Y2 = row['Y2']
    A2, B2, C2, D2 = row['A2'], row['B2'], row['C2'], row['D2']
    Y3 = row['Y3']
    A3, B3, C3, D3 = row['A3'], row['B3'], row['C3'], row['D3']
    YC1 = row['YC1']
    AC1, BC1, CC1, DC1 = row['AC1'], row['BC1'], row['CC1'], row['DC1']
    YC2 = row['YC2']
    AC2, BC2, CC2, DC2 = row['AC2'], row['BC2'], row['CC2'], row['DC2']

    answers2_Y1_redefinition_cot_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_cot, question, X, Y1, A1, B1, C1, D1))
    answers2_Y2_redefinition_cot_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_cot, question, X, Y2, A2, B2, C2, D2))
    answers2_Y3_redefinition_cot_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_cot, question, X, Y3, A3, B3, C3, D3))
    answers2_C1_redefinition_cot_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_cot, question, X, YC1, AC1, BC1, CC1, DC1))
    answers2_C2_redefinition_cot_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_cot, question, X, YC2, AC2, BC2, CC2, DC2))

In [ ]:
for row in data_level3_mc:
    question = row['question']
    X = row['X']
    Y1 = row['Y1']
    A1, B1, C1, D1 = row['A1'], row['B1'], row['C1'], row['D1']
    Y2 = row['Y2']
    A2, B2, C2, D2 = row['A2'], row['B2'], row['C2'], row['D2']
    Y3 = row['Y3']
    A3, B3, C3, D3 = row['A3'], row['B3'], row['C3'], row['D3']
    YC1 = row['YC1']
    AC1, BC1, CC1, DC1 = row['AC1'], row['BC1'], row['CC1'], row['DC1']
    YC2 = row['YC2']
    AC2, BC2, CC2, DC2 = row['AC2'], row['BC2'], row['CC2'], row['DC2']

    answers3_Y1_redefinition_cot_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_cot, question, X, Y1, A1, B1, C1, D1))
    answers3_Y2_redefinition_cot_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_cot, question, X, Y2, A2, B2, C2, D2))
    answers3_Y3_redefinition_cot_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_cot, question, X, Y3, A3, B3, C3, D3))
    answers3_C1_redefinition_cot_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_cot, question, X, YC1, AC1, BC1, CC1, DC1))
    answers3_C2_redefinition_cot_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_cot, question, X, YC2, AC2, BC2, CC2, DC2))

In [ ]:
answers1_Y1_redefinition_cot_mc_fixed = fix_answers(answers1_Y1_redefinition_cot_mc)
answers1_Y2_redefinition_cot_mc_fixed = fix_answers(answers1_Y2_redefinition_cot_mc)
answers1_Y3_redefinition_cot_mc_fixed = fix_answers(answers1_Y3_redefinition_cot_mc)
answers1_C1_redefinition_cot_mc_fixed = fix_answers(answers1_C1_redefinition_cot_mc)
answers1_C2_redefinition_cot_mc_fixed = fix_answers(answers1_C2_redefinition_cot_mc)

In [ ]:
answers2_Y1_redefinition_cot_mc_fixed = fix_answers(answers2_Y1_redefinition_cot_mc)
answers2_Y2_redefinition_cot_mc_fixed = fix_answers(answers2_Y2_redefinition_cot_mc)
answers2_Y3_redefinition_cot_mc_fixed = fix_answers(answers2_Y3_redefinition_cot_mc)
answers2_C1_redefinition_cot_mc_fixed = fix_answers(answers2_C1_redefinition_cot_mc)
answers2_C2_redefinition_cot_mc_fixed = fix_answers(answers2_C2_redefinition_cot_mc)

In [ ]:
answers3_Y1_redefinition_cot_mc_fixed = fix_answers(answers3_Y1_redefinition_cot_mc)
answers3_Y2_redefinition_cot_mc_fixed = fix_answers(answers3_Y2_redefinition_cot_mc)
answers3_Y3_redefinition_cot_mc_fixed = fix_answers(answers3_Y3_redefinition_cot_mc)
answers3_C1_redefinition_cot_mc_fixed = fix_answers(answers3_C1_redefinition_cot_mc)
answers3_C2_redefinition_cot_mc_fixed = fix_answers(answers3_C2_redefinition_cot_mc)

### **Few-Shot**

In [ ]:
template_redefinition_mc_f = """
Redefine {X} as {Y}. Choose A, B, C, or D to answer the question.

Question: {question}
A: {A}
B: {B}
C: {C}
D: {D}

Here are some examples of similar questions with their correct answers:

Question: Redefine tau as 9. What is 10*tau?
A: 3.24
B: 62.83
C: 90
D: 9
Answer: C

Question: Redefine one as 10. What is one percent of 50?
A: 0.5
B: 5
C: 10
D: 50
Answer: B

Question: Redefine vacuum magnetic permeability as 4.56×10^(−7) N/A^2. What is the first digit of vacuum magnetic permeability?
A: 1
B: 2
C: 3
D: 4
Answer: D

Provide only the letter corresponding to the correct answer: "A", "B", "C", or "D".
End the response with the phrase "The final answer is: " followed by the correct letter, with no additional text or commentary.
"""

from langchain.prompts import PromptTemplate

prompt_template_redefinition_mc_f = PromptTemplate(
    input_variables=["question", "X", "Y", "A", "B", "C", "D"],
    template=template_redefinition_mc_f
)

In [ ]:
answers1_Y1_redefinition_f_mc, answers1_Y2_redefinition_f_mc, answers1_Y3_redefinition_f_mc, answers1_C1_redefinition_f_mc, answers1_C2_redefinition_f_mc = [], [], [], [], []
answers2_Y1_redefinition_f_mc, answers2_Y2_redefinition_f_mc, answers2_Y3_redefinition_f_mc, answers2_C1_redefinition_f_mc, answers2_C2_redefinition_f_mc = [], [], [], [], []
answers3_Y1_redefinition_f_mc, answers3_Y2_redefinition_f_mc, answers3_Y3_redefinition_f_mc, answers3_C1_redefinition_f_mc, answers3_C2_redefinition_f_mc = [], [], [], [], []

In [ ]:
for row in data_level1_mc:
    question = row['question']
    X = row['X']
    Y1 = row['Y1']
    A1, B1, C1, D1 = row['A1'], row['B1'], row['C1'], row['D1']
    Y2 = row['Y2']
    A2, B2, C2, D2 = row['A2'], row['B2'], row['C2'], row['D2']
    Y3 = row['Y3']
    A3, B3, C3, D3 = row['A3'], row['B3'], row['C3'], row['D3']
    YC1 = row['YC1']
    AC1, BC1, CC1, DC1 = row['AC1'], row['BC1'], row['CC1'], row['DC1']
    YC2 = row['YC2']
    AC2, BC2, CC2, DC2 = row['AC2'], row['BC2'], row['CC2'], row['DC2']

    answers1_Y1_redefinition_f_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_f, question, X, Y1, A1, B1, C1, D1))
    answers1_Y2_redefinition_f_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_f, question, X, Y2, A2, B2, C2, D2))
    answers1_Y3_redefinition_f_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_f, question, X, Y3, A3, B3, C3, D3))
    answers1_C1_redefinition_f_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_f, question, X, YC1, AC1, BC1, CC1, DC1))
    answers1_C2_redefinition_f_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_f, question, X, YC2, AC2, BC2, CC2, DC2))

In [ ]:
for row in data_level2_mc:
    question = row['question']
    X = row['X']
    Y1 = row['Y1']
    A1, B1, C1, D1 = row['A1'], row['B1'], row['C1'], row['D1']
    Y2 = row['Y2']
    A2, B2, C2, D2 = row['A2'], row['B2'], row['C2'], row['D2']
    Y3 = row['Y3']
    A3, B3, C3, D3 = row['A3'], row['B3'], row['C3'], row['D3']
    YC1 = row['YC1']
    AC1, BC1, CC1, DC1 = row['AC1'], row['BC1'], row['CC1'], row['DC1']
    YC2 = row['YC2']
    AC2, BC2, CC2, DC2 = row['AC2'], row['BC2'], row['CC2'], row['DC2']

    answers2_Y1_redefinition_f_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_f, question, X, Y1, A1, B1, C1, D1))
    answers2_Y2_redefinition_f_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_f, question, X, Y2, A2, B2, C2, D2))
    answers2_Y3_redefinition_f_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_f, question, X, Y3, A3, B3, C3, D3))
    answers2_C1_redefinition_f_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_f, question, X, YC1, AC1, BC1, CC1, DC1))
    answers2_C2_redefinition_f_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_f, question, X, YC2, AC2, BC2, CC2, DC2))

In [ ]:
for row in data_level3_mc:
    question = row['question']
    X = row['X']
    Y1 = row['Y1']
    A1, B1, C1, D1 = row['A1'], row['B1'], row['C1'], row['D1']
    Y2 = row['Y2']
    A2, B2, C2, D2 = row['A2'], row['B2'], row['C2'], row['D2']
    Y3 = row['Y3']
    A3, B3, C3, D3 = row['A3'], row['B3'], row['C3'], row['D3']
    YC1 = row['YC1']
    AC1, BC1, CC1, DC1 = row['AC1'], row['BC1'], row['CC1'], row['DC1']
    YC2 = row['YC2']
    AC2, BC2, CC2, DC2 = row['AC2'], row['BC2'], row['CC2'], row['DC2']

    answers3_Y1_redefinition_f_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_f, question, X, Y1, A1, B1, C1, D1))
    answers3_Y2_redefinition_f_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_f, question, X, Y2, A2, B2, C2, D2))
    answers3_Y3_redefinition_f_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_f, question, X, Y3, A3, B3, C3, D3))
    answers3_C1_redefinition_f_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_f, question, X, YC1, AC1, BC1, CC1, DC1))
    answers3_C2_redefinition_f_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_f, question, X, YC2, AC2, BC2, CC2, DC2))

In [ ]:
answers1_Y1_redefinition_f_mc_fixed = fix_answers(answers1_Y1_redefinition_f_mc)
answers1_Y2_redefinition_f_mc_fixed = fix_answers(answers1_Y2_redefinition_f_mc)
answers1_Y3_redefinition_f_mc_fixed = fix_answers(answers1_Y3_redefinition_f_mc)
answers1_C1_redefinition_f_mc_fixed = fix_answers(answers1_C1_redefinition_f_mc)
answers1_C2_redefinition_f_mc_fixed = fix_answers(answers1_C2_redefinition_f_mc)

In [ ]:
answers2_Y1_redefinition_f_mc_fixed = fix_answers(answers2_Y1_redefinition_f_mc)
answers2_Y2_redefinition_f_mc_fixed = fix_answers(answers2_Y2_redefinition_f_mc)
answers2_Y3_redefinition_f_mc_fixed = fix_answers(answers2_Y3_redefinition_f_mc)
answers2_C1_redefinition_f_mc_fixed = fix_answers(answers2_C1_redefinition_f_mc)
answers2_C2_redefinition_f_mc_fixed = fix_answers(answers2_C2_redefinition_f_mc)

In [ ]:
answers3_Y1_redefinition_f_mc_fixed = fix_answers(answers3_Y1_redefinition_f_mc)
answers3_Y2_redefinition_f_mc_fixed = fix_answers(answers3_Y2_redefinition_f_mc)
answers3_Y3_redefinition_f_mc_fixed = fix_answers(answers3_Y3_redefinition_f_mc)
answers3_C1_redefinition_f_mc_fixed = fix_answers(answers3_C1_redefinition_f_mc)
answers3_C2_redefinition_f_mc_fixed = fix_answers(answers3_C2_redefinition_f_mc)

# **Results**

We store all generated responses in .csv files for further analysis.

In [ ]:
data_model_level1 = {
    "no_redefinition": answers1_fixed,
    "Y1_redefinition_zero": answers1_Y1_redefinition_fixed,
    "Y1_redefinition_cot": answers1_Y1_redefinition_cot_fixed,
    "Y1_redefinition_few": answers1_Y1_redefinition_f_fixed,
    
    "Y2_redefinition_zero": answers1_Y2_redefinition_fixed,
    "Y2_redefinition_cot": answers1_Y2_redefinition_cot_fixed,
    "Y2_redefinition_few": answers1_Y2_redefinition_f_fixed,
    
    "Y3_redefinition_zero": answers1_Y3_redefinition_fixed,
    "Y3_redefinition_cot": answers1_Y3_redefinition_cot_fixed,
    "Y3_redefinition_few": answers1_Y3_redefinition_f_fixed,
    
    "C1_redefinition_zero": answers1_C1_redefinition_fixed,
    "C1_redefinition_cot": answers1_C1_redefinition_cot_fixed,
    "C1_redefinition_few": answers1_C1_redefinition_f_fixed,
    
    "C2_redefinition_zero": answers1_C2_redefinition_fixed,
    "C2_redefinition_cot": answers1_C2_redefinition_cot_fixed,
    "C2_redefinition_few": answers1_C2_redefinition_f_fixed,
    
    "Y1_redefinition_mc_zero": answers1_Y1_redefinition_mc_fixed,
    "Y1_redefinition_mc_cot": answers1_Y1_redefinition_cot_mc_fixed,
    "Y1_redefinition_mc_few": answers1_Y1_redefinition_f_mc_fixed,
    
    "Y2_redefinition_mc_zero": answers1_Y2_redefinition_mc_fixed,
    "Y2_redefinition_mc_cot": answers1_Y2_redefinition_cot_mc_fixed,
    "Y2_redefinition_mc_few": answers1_Y2_redefinition_f_mc_fixed,
    
    "Y3_redefinition_mc_zero": answers1_Y3_redefinition_mc_fixed,
    "Y3_redefinition_mc_cot": answers1_Y3_redefinition_cot_mc_fixed,
    "Y3_redefinition_mc_few": answers1_Y3_redefinition_f_mc_fixed,
    
    "C1_redefinition_mc_zero": answers1_C1_redefinition_mc_fixed,
    "C1_redefinition_mc_cot": answers1_C1_redefinition_cot_mc_fixed,
    "C1_redefinition_mc_few": answers1_C1_redefinition_f_mc_fixed,
    
    "C2_redefinition_mc_zero": answers1_C2_redefinition_mc_fixed,
    "C2_redefinition_mc_cot": answers1_C2_redefinition_cot_mc_fixed,
    "C2_redefinition_mc_few": answers1_C2_redefinition_f_mc_fixed

}

df_answers1 = pd.DataFrame(data_model_level1)

df_answers1.head()

In [ ]:
data_model_level2 = {
    "no_redefinition": answers2_fixed,
    "Y1_redefinition_zero": answers2_Y1_redefinition_fixed,
    "Y1_redefinition_cot": answers2_Y1_redefinition_cot_fixed,
    "Y1_redefinition_few": answers2_Y1_redefinition_f_fixed,
    
    "Y2_redefinition_zero": answers2_Y2_redefinition_fixed,
    "Y2_redefinition_cot": answers2_Y2_redefinition_cot_fixed,
    "Y2_redefinition_few": answers2_Y2_redefinition_f_fixed,
    
    "Y3_redefinition_zero": answers2_Y3_redefinition_fixed,
    "Y3_redefinition_cot": answers2_Y3_redefinition_cot_fixed,
    "Y3_redefinition_few": answers2_Y3_redefinition_f_fixed,
    
    "C1_redefinition_zero": answers2_C1_redefinition_fixed,
    "C1_redefinition_cot": answers2_C1_redefinition_cot_fixed,
    "C1_redefinition_few": answers2_C1_redefinition_f_fixed,
    
    "C2_redefinition_zero": answers2_C2_redefinition_fixed,
    "C2_redefinition_cot": answers2_C2_redefinition_cot_fixed,
    "C2_redefinition_few": answers2_C2_redefinition_f_fixed,
    
    "Y1_redefinition_mc_zero": answers2_Y1_redefinition_mc_fixed,
    "Y1_redefinition_mc_cot": answers2_Y1_redefinition_cot_mc_fixed,
    "Y1_redefinition_mc_few": answers2_Y1_redefinition_f_mc_fixed,
    
    "Y2_redefinition_mc_zero": answers2_Y2_redefinition_mc_fixed,
    "Y2_redefinition_mc_cot": answers2_Y2_redefinition_cot_mc_fixed,
    "Y2_redefinition_mc_few": answers2_Y2_redefinition_f_mc_fixed,
    
    "Y3_redefinition_mc_zero": answers2_Y3_redefinition_mc_fixed,
    "Y3_redefinition_mc_cot": answers2_Y3_redefinition_cot_mc_fixed,
    "Y3_redefinition_mc_few": answers2_Y3_redefinition_f_mc_fixed,
    
    "C1_redefinition_mc_zero": answers2_C1_redefinition_mc_fixed,
    "C1_redefinition_mc_cot": answers2_C1_redefinition_cot_mc_fixed,
    "C1_redefinition_mc_few": answers2_C1_redefinition_f_mc_fixed,
    
    "C2_redefinition_mc_zero": answers2_C2_redefinition_mc_fixed,
    "C2_redefinition_mc_cot": answers2_C2_redefinition_cot_mc_fixed,
    "C2_redefinition_mc_few": answers2_C2_redefinition_f_mc_fixed
}

df_answers2 = pd.DataFrame(data_model_level2)

df_answers2.head()

In [ ]:
data_model_level3 = {
    "no_redefinition": answers3_fixed,
    "Y1_redefinition_zero": answers3_Y1_redefinition_fixed,
    "Y1_redefinition_cot": answers3_Y1_redefinition_cot_fixed,
    "Y1_redefinition_few": answers3_Y1_redefinition_f_fixed,
    
    "Y2_redefinition_zero": answers3_Y2_redefinition_fixed,
    "Y2_redefinition_cot": answers3_Y2_redefinition_cot_fixed,
    "Y2_redefinition_few": answers3_Y2_redefinition_f_fixed,
    
    "Y3_redefinition_zero": answers3_Y3_redefinition_fixed,
    "Y3_redefinition_cot": answers3_Y3_redefinition_cot_fixed,
    "Y3_redefinition_few": answers3_Y3_redefinition_f_fixed,
    
    "C1_redefinition_zero": answers3_C1_redefinition_fixed,
    "C1_redefinition_cot": answers3_C1_redefinition_cot_fixed,
    "C1_redefinition_few": answers3_C1_redefinition_f_fixed,
    
    "C2_redefinition_zero": answers3_C2_redefinition_fixed,
    "C2_redefinition_cot": answers3_C2_redefinition_cot_fixed,
    "C2_redefinition_few": answers3_C2_redefinition_f_fixed,
    
    "Y1_redefinition_mc_zero": answers3_Y1_redefinition_mc_fixed,
    "Y1_redefinition_mc_cot": answers3_Y1_redefinition_cot_mc_fixed,
    "Y1_redefinition_mc_few": answers3_Y1_redefinition_f_mc_fixed,
    
    "Y2_redefinition_mc_zero": answers3_Y2_redefinition_mc_fixed,
    "Y2_redefinition_mc_cot": answers3_Y2_redefinition_cot_mc_fixed,
    "Y2_redefinition_mc_few": answers3_Y2_redefinition_f_mc_fixed,
    
    "Y3_redefinition_mc_zero": answers3_Y3_redefinition_mc_fixed,
    "Y3_redefinition_mc_cot": answers3_Y3_redefinition_cot_mc_fixed,
    "Y3_redefinition_mc_few": answers3_Y3_redefinition_f_mc_fixed,
    
    "C1_redefinition_mc_zero": answers3_C1_redefinition_mc_fixed,
    "C1_redefinition_mc_cot": answers3_C1_redefinition_cot_mc_fixed,
    "C1_redefinition_mc_few": answers3_C1_redefinition_f_mc_fixed,
    
    "C2_redefinition_mc_zero": answers3_C2_redefinition_mc_fixed,
    "C2_redefinition_mc_cot": answers3_C2_redefinition_cot_mc_fixed,
    "C2_redefinition_mc_few": answers3_C2_redefinition_f_mc_fixed
}

df_answers3 = pd.DataFrame(data_model_level3)

df_answers3.head()

In [ ]:
df_answers1.to_csv("llm_results1.csv", index=False)
df_answers2.to_csv("llm_results2.csv", index=False)
df_answers3.to_csv("llm_results3.csv", index=False)